In [1]:
# Cell 1 — Environment setup
import os, json, random, subprocess
import numpy as np
import torch
from google.colab import drive

drive.mount("/content/drive")

# ── Reproducibility ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Device ──
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU:    {torch.cuda.get_device_name(0)}")

# ── Install pinned transformers ──
subprocess.run(["pip", "install", "transformers==4.40.0", "-q"])
print("transformers==4.40.0 ready.")

# ── Drive directory structure ──
BASE_DIR       = "/content/drive/MyDrive/BioBERT_Project/biobert_pneumonia"
DATA_DIR       = os.path.join(BASE_DIR, "data")
CHECKPOINT_DIR = os.path.join(BASE_DIR, "checkpoints")
RESULTS_DIR    = os.path.join(BASE_DIR, "results")

for d in [DATA_DIR, CHECKPOINT_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

# ── All hyperparameters in one place ──
CONFIG = {
    "seed"           : SEED,
    "model_name"     : "dmis-lab/biobert-base-cased-v1.1",
    "max_length"     : 128,
    "batch_size"     : 16,
    "learning_rate"  : 2e-5,
    "num_epochs"     : 5,
    "warmup_ratio"   : 0.1,
    "weight_decay"   : 0.01,
    "primary_metric" : "roc_auc",
    "num_classes"    : 2,
    "base_dir"       : BASE_DIR,
    "data_dir"       : DATA_DIR,
    "checkpoint_dir" : CHECKPOINT_DIR,
    "results_dir"    : RESULTS_DIR,
}

print(f"\nBase:        {BASE_DIR}")
print(f"Data:        {DATA_DIR}")
print(f"Checkpoints: {CHECKPOINT_DIR}")
print(f"Results:     {RESULTS_DIR}")
print("\nCell 1 complete.")

Mounted at /content/drive
Device: cuda
GPU:    Tesla T4
transformers==4.40.0 ready.

Base:        /content/drive/MyDrive/BioBERT_Project/biobert_pneumonia
Data:        /content/drive/MyDrive/BioBERT_Project/biobert_pneumonia/data
Checkpoints: /content/drive/MyDrive/BioBERT_Project/biobert_pneumonia/checkpoints
Results:     /content/drive/MyDrive/BioBERT_Project/biobert_pneumonia/results

Cell 1 complete.


In [3]:
# Cell 2 — Load dataset, binary mapping, duplicate and conflict analysis
import pandas as pd
import os

# ── Load single dataset file ──
DATASET_PATH = os.path.join(DATA_DIR, "BioBERT_Text_Dataset_Encoded.csv")

df = pd.read_csv(DATASET_PATH)
print(f"Dataset loaded: {len(df)} rows, columns: {df.columns.tolist()}")

# ── Binary mapping ──
df["text_clean"]   = df["text"].str.strip().str.lower()
df["binary_label"] = (df["disease"].str.strip().str.lower() == "pneumonia").astype(int)

print(f"\nBinary distribution:")
print(f"  Pneumonia     (1): {(df['binary_label']==1).sum()}")
print(f"  Non-pneumonia (0): {(df['binary_label']==0).sum()}")
print(f"  Imbalance ratio:   {(df['binary_label']==0).sum() / (df['binary_label']==1).sum():.2f}:1")

# ── Duplicate and conflict analysis ──
# For each unique text, collect the set of binary labels it appears under
text_to_labels = df.groupby("text_clean")["binary_label"].apply(set)

# Genuine binary conflict = same text maps to BOTH 0 and 1
conflict_texts   = text_to_labels[text_to_labels.apply(lambda s: len(s) > 1)]
duplicate_texts  = text_to_labels[text_to_labels.index.isin(
                       df[df.duplicated("text_clean", keep=False)]["text_clean"])]

conflict_rows    = df[df["text_clean"].isin(conflict_texts.index)]
duplicate_rows   = df[df["text_clean"].isin(duplicate_texts.index)]

print(f"\nDuplicate and conflict analysis:")
print(f"  Total unique texts:                 {df['text_clean'].nunique()}")
print(f"  Texts appearing more than once:     {duplicate_texts.index.nunique()}")
print(f"  Genuine binary conflicts (0 and 1): {len(conflict_texts)}")
print(f"  Rows affected by conflicts:         {len(conflict_rows)}")
print(f"  Non-conflicting duplicate rows:     {len(duplicate_rows) - len(conflict_rows)}")

if len(conflict_texts) > 0:
    print(f"\n  Sample conflicting texts:")
    for txt, labels in list(conflict_texts.items())[:3]:
        diseases = df[df["text_clean"] == txt]["disease"].unique().tolist()
        print(f"    Binary labels: {labels} | Diseases: {diseases}")
        print(f"    Text: '{txt[:80]}'")

print("\nCell 2 complete.")

Dataset loaded: 3766 rows, columns: ['text', 'disease', 'label']

Binary distribution:
  Pneumonia     (1): 1212
  Non-pneumonia (0): 2554
  Imbalance ratio:   2.11:1

Duplicate and conflict analysis:
  Total unique texts:                 3046
  Texts appearing more than once:     487
  Genuine binary conflicts (0 and 1): 12
  Rows affected by conflicts:         27
  Non-conflicting duplicate rows:     1180

  Sample conflicting texts:
    Binary labels: {0, 1} | Diseases: ['pulmonary fibrosis', 'pneumonia']
    Text: 'the patient presents with cough and fever.'
    Binary labels: {0, 1} | Diseases: ['pulmonary congestion', 'pneumonia']
    Text: 'the patient presents with sharp chest pain and cough.'
    Binary labels: {0, 1} | Diseases: ['pneumonia', 'pleural effusion']
    Text: 'the patient presents with sharp chest pain, cough, and difficulty breathing.'

Cell 2 complete.


In [4]:
# Cell 3 — Remove genuine conflicts, group-aware stratified split
from sklearn.model_selection import StratifiedGroupKFold
import numpy as np

# ── Remove 27 genuine binary conflict rows ──
conflict_mask = df["text_clean"].isin(conflict_texts.index)
df_clean      = df[~conflict_mask].copy().reset_index(drop=True)

print(f"Rows removed (genuine binary conflicts): {conflict_mask.sum()}")
print(f"Remaining rows:                          {len(df_clean)}")
print(f"  Pneumonia     (1): {(df_clean['binary_label']==1).sum()}")
print(f"  Non-pneumonia (0): {(df_clean['binary_label']==0).sum()}")

# ── Assign group ID — identical texts share one group ──
# This ensures the same text never appears in two different splits
unique_texts     = df_clean["text_clean"].unique()
text_to_group    = {t: i for i, t in enumerate(unique_texts)}
df_clean["group"] = df_clean["text_clean"].map(text_to_group)

print(f"\nTotal unique text groups: {df_clean['group'].nunique()}")

# ── Group-aware stratified split: 70% train / 15% val / 15% test ──
# Strategy: use group-level binary label (majority within group,
# but since non-conflicting groups are all-same-label this is exact)
group_df = (df_clean.groupby("group")["binary_label"]
            .first().reset_index()
            .rename(columns={"binary_label": "group_label"}))

groups       = df_clean["group"].values
labels       = df_clean["binary_label"].values

# First split: 70% train, 30% temp
sgkf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=SEED)

# Use one fold to get ~70/30 split
for train_idx, temp_idx in sgkf.split(df_clean, labels, groups):
    break   # take the first fold: ~90/10, we then re-split temp

# Re-split temp into val and test (50/50 of the 30%)
temp_df      = df_clean.iloc[temp_idx].copy()
temp_groups  = temp_df["group"].values
temp_labels  = temp_df["binary_label"].values

sgkf2 = StratifiedGroupKFold(n_splits=2, shuffle=True, random_state=SEED)
for val_idx, test_idx in sgkf2.split(temp_df, temp_labels, temp_groups):
    break

train_df = df_clean.iloc[train_idx].copy().reset_index(drop=True)
val_df   = temp_df.iloc[val_idx].copy().reset_index(drop=True)
test_df  = temp_df.iloc[test_idx].copy().reset_index(drop=True)

# ── Verify no text leakage across splits ──
train_texts = set(train_df["text_clean"])
val_texts   = set(val_df["text_clean"])
test_texts  = set(test_df["text_clean"])

train_val_leak  = train_texts & val_texts
train_test_leak = train_texts & test_texts
val_test_leak   = val_texts   & test_texts

print(f"\nSplit sizes:")
print(f"  Train: {len(train_df):>5} rows | "
      f"Pneumonia: {(train_df['binary_label']==1).sum()} | "
      f"Non-pneumonia: {(train_df['binary_label']==0).sum()}")
print(f"  Val:   {len(val_df):>5} rows | "
      f"Pneumonia: {(val_df['binary_label']==1).sum()} | "
      f"Non-pneumonia: {(val_df['binary_label']==0).sum()}")
print(f"  Test:  {len(test_df):>5} rows | "
      f"Pneumonia: {(test_df['binary_label']==1).sum()} | "
      f"Non-pneumonia: {(test_df['binary_label']==0).sum()}")

print(f"\nLeakage check:")
print(f"  Train ∩ Val:  {len(train_val_leak)}  texts")
print(f"  Train ∩ Test: {len(train_test_leak)} texts")
print(f"  Val   ∩ Test: {len(val_test_leak)}  texts")

# ── Save splits to Drive ──
train_df.to_csv(os.path.join(DATA_DIR, "train_binary.csv"), index=False)
val_df.to_csv(os.path.join(DATA_DIR,   "val_binary.csv"),   index=False)
test_df.to_csv(os.path.join(DATA_DIR,  "test_binary.csv"),  index=False)

print(f"\nSplits saved to {DATA_DIR}")
print("Cell 3 complete.")

Rows removed (genuine binary conflicts): 27
Remaining rows:                          3739
  Pneumonia     (1): 1200
  Non-pneumonia (0): 2539

Total unique text groups: 3034

Split sizes:
  Train:  3349 rows | Pneumonia: 1082 | Non-pneumonia: 2267
  Val:     200 rows | Pneumonia: 60 | Non-pneumonia: 140
  Test:    190 rows | Pneumonia: 58 | Non-pneumonia: 132

Leakage check:
  Train ∩ Val:  0  texts
  Train ∩ Test: 0 texts
  Val   ∩ Test: 0  texts

Splits saved to /content/drive/MyDrive/BioBERT_Project/biobert_pneumonia/data
Cell 3 complete.


In [5]:
# Cell 4 — BioBERT tokenizer and PyTorch dataset
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer

# ── Load tokenizer ──
tokenizer = BertTokenizer.from_pretrained(CONFIG["model_name"])
print(f"Tokenizer loaded: {CONFIG['model_name']}")

# ── Check actual token lengths in training data ──
sample_lengths = train_df["text"].apply(
    lambda t: len(tokenizer.encode(t, add_special_tokens=True))
)
print(f"\nToken length stats (train):")
print(f"  Max:    {sample_lengths.max()}")
print(f"  Mean:   {sample_lengths.mean():.1f}")
print(f"  95th %: {int(np.percentile(sample_lengths, 95))}")
print(f"  Using max_length: {CONFIG['max_length']} "
      f"({'sufficient' if sample_lengths.max() <= CONFIG['max_length'] else 'some truncation will occur'})")

# ── Dataset class ──
class PneumoniaDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length):
        self.texts     = dataframe["text"].astype(str).tolist()
        self.labels    = dataframe["binary_label"].astype(int).tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids":      encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label":          torch.tensor(self.labels[idx], dtype=torch.long)
        }

# ── Create datasets and dataloaders ──
train_dataset = PneumoniaDataset(train_df, tokenizer, CONFIG["max_length"])
val_dataset   = PneumoniaDataset(val_df,   tokenizer, CONFIG["max_length"])
test_dataset  = PneumoniaDataset(test_df,  tokenizer, CONFIG["max_length"])

train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"],
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=CONFIG["batch_size"],
                          shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=CONFIG["batch_size"],
                          shuffle=False, num_workers=2, pin_memory=True)

print(f"\nDatasets:")
print(f"  Train: {len(train_dataset)} samples, {len(train_loader)} batches")
print(f"  Val:   {len(val_dataset)} samples, {len(val_loader)} batches")
print(f"  Test:  {len(test_dataset)} samples, {len(test_loader)} batches")

print("\nCell 4 complete.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.txt: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Tokenizer loaded: dmis-lab/biobert-base-cased-v1.1

Token length stats (train):
  Max:    53
  Mean:   26.3
  95th %: 40
  Using max_length: 128 (sufficient)

Datasets:
  Train: 3349 samples, 210 batches
  Val:   200 samples, 13 batches
  Test:  190 samples, 12 batches

Cell 4 complete.


In [6]:
# Cell 5 — Model, class weights, optimizer, scheduler, training functions
import torch.nn as nn
from transformers import BertModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import (roc_auc_score, f1_score, accuracy_score,
                              precision_score, recall_score, confusion_matrix)

# ── Model ──
class BioBERTBinaryClassifier(nn.Module):
    """
    BioBERT backbone with a 2-class classification head.
    Uses [CLS] token representation for sequence classification.
    """
    def __init__(self, model_name, dropout_rate=0.3):
        super().__init__()
        self.bert       = BertModel.from_pretrained(model_name)
        self.dropout    = nn.Dropout(dropout_rate)
        self.classifier = nn.Linear(self.bert.config.hidden_size, 2)

    def forward(self, input_ids, attention_mask):
        outputs    = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_vector = outputs.last_hidden_state[:, 0, :]
        return self.classifier(self.dropout(cls_vector))

model = BioBERTBinaryClassifier(CONFIG["model_name"]).to(device)
print(f"Model loaded on {device}")
print(f"  Total parameters:     {sum(p.numel() for p in model.parameters()):,}")
print(f"  Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# ── Class weights (computed from train set only) ──
n_neg  = (train_df["binary_label"] == 0).sum()
n_pos  = (train_df["binary_label"] == 1).sum()
w_neg  = len(train_df) / (2 * n_neg)
w_pos  = len(train_df) / (2 * n_pos)
class_weights = torch.tensor([w_neg, w_pos], dtype=torch.float32).to(device)
print(f"\nClass weights — Non-pneumonia: {w_neg:.4f} | Pneumonia: {w_pos:.4f}")

criterion = nn.CrossEntropyLoss(weight=class_weights)

# ── Optimizer with weight decay excluding bias and LayerNorm ──
no_decay = ["bias", "LayerNorm.weight"]
optimizer_params = [
    {"params": [p for n, p in model.named_parameters()
                if not any(nd in n for nd in no_decay)], "weight_decay": CONFIG["weight_decay"]},
    {"params": [p for n, p in model.named_parameters()
                if any(nd in n for nd in no_decay)],     "weight_decay": 0.0},
]
optimizer = AdamW(optimizer_params, lr=CONFIG["learning_rate"])

# ── Scheduler ──
total_steps  = len(train_loader) * CONFIG["num_epochs"]
warmup_steps = int(CONFIG["warmup_ratio"] * total_steps)
scheduler    = get_linear_schedule_with_warmup(optimizer,
                   num_warmup_steps=warmup_steps,
                   num_training_steps=total_steps)

print(f"\nOptimizer: AdamW | LR: {CONFIG['learning_rate']} | "
      f"Total steps: {total_steps} | Warmup: {warmup_steps}")

# ── Training function ──
def train_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for batch in loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["label"].to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss   = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        correct    += (logits.argmax(dim=1) == labels).sum().item()
        total      += labels.size(0)

    return total_loss / len(loader), correct / total

# ── Evaluation function ──
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss  = 0.0
    all_preds, all_probs, all_labels = [], [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["label"].to(device)

            logits = model(input_ids, attention_mask)
            loss   = criterion(logits, labels)
            probs  = torch.softmax(logits, dim=1)[:, 1]

            total_loss  += loss.item()
            all_preds.extend(logits.argmax(dim=1).cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    metrics  = {
        "loss"       : round(avg_loss, 6),
        "accuracy"   : round(accuracy_score(all_labels, all_preds), 4),
        "precision"  : round(precision_score(all_labels, all_preds, zero_division=0), 4),
        "recall"     : round(recall_score(all_labels, all_preds, zero_division=0), 4),
        "f1"         : round(f1_score(all_labels, all_preds, zero_division=0), 4),
        "roc_auc"    : round(roc_auc_score(all_labels, all_probs), 4),
        "specificity": round(confusion_matrix(all_labels, all_preds)[0, 0] /
                             confusion_matrix(all_labels, all_preds)[0].sum(), 4),
    }
    return metrics, all_preds, all_probs, all_labels

print("\nCell 5 complete.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Model loaded on cuda
  Total parameters:     108,311,810
  Trainable parameters: 108,311,810

Class weights — Non-pneumonia: 0.7386 | Pneumonia: 1.5476

Optimizer: AdamW | LR: 2e-05 | Total steps: 1050 | Warmup: 105

Cell 5 complete.


In [7]:
#full training loop

In [8]:
# Cell 6 — Training loop with checkpointing and early stopping
import time, json

BEST_MODEL_PATH  = os.path.join(CHECKPOINT_DIR, "best_model.pt")
HISTORY_PATH     = os.path.join(RESULTS_DIR,    "training_history.json")
EARLY_STOP_PATIENCE = 3   # stop if primary metric does not improve for 3 epochs

history          = []
best_metric      = -1.0
best_epoch       = -1
no_improve_count = 0

print("=" * 60)
print("TRAINING — BioBERT Binary Pneumonia Classifier")
print("=" * 60)
print(f"  Epochs:          {CONFIG['num_epochs']}")
print(f"  Train batches:   {len(train_loader)}")
print(f"  Primary metric:  {CONFIG['primary_metric']}")
print(f"  Early stopping:  patience={EARLY_STOP_PATIENCE}")
print("=" * 60)

for epoch in range(1, CONFIG["num_epochs"] + 1):
    t0 = time.time()

    train_loss, train_acc = train_epoch(
        model, train_loader, optimizer, scheduler, criterion, device
    )
    val_metrics, _, _, _ = evaluate(
        model, val_loader, criterion, device
    )

    elapsed = time.time() - t0
    primary = val_metrics[CONFIG["primary_metric"]]

    epoch_record = {
        "epoch"      : epoch,
        "train_loss" : round(train_loss, 6),
        "train_acc"  : round(train_acc,  4),
        **{f"val_{k}": v for k, v in val_metrics.items()},
    }
    history.append(epoch_record)

    print(f"\nEpoch {epoch}/{CONFIG['num_epochs']}  ({elapsed:.0f}s)")
    print(f"  Train  — Loss: {train_loss:.4f} | Acc: {train_acc*100:.2f}%")
    print(f"  Val    — Loss: {val_metrics['loss']:.4f} | "
          f"Acc: {val_metrics['accuracy']*100:.2f}% | "
          f"Recall: {val_metrics['recall']:.4f} | "
          f"F1: {val_metrics['f1']:.4f} | "
          f"AUC: {val_metrics['roc_auc']:.4f}")

    # ── Save best checkpoint ──
    if primary > best_metric:
        best_metric      = primary
        best_epoch       = epoch
        no_improve_count = 0

        torch.save({
            "epoch"               : epoch,
            "model_state_dict"    : model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_metrics"         : val_metrics,
            "config"              : CONFIG,
        }, BEST_MODEL_PATH)
        print(f"  ✓ Best model saved — Val {CONFIG['primary_metric'].upper()}: {primary:.4f}")
    else:
        no_improve_count += 1
        print(f"  No improvement ({no_improve_count}/{EARLY_STOP_PATIENCE}) — "
              f"Best so far: {best_metric:.4f} (Epoch {best_epoch})")

    # ── Save history after every epoch ──
    with open(HISTORY_PATH, "w") as f:
        json.dump(history, f, indent=2)

    if no_improve_count >= EARLY_STOP_PATIENCE:
        print(f"\nEarly stopping triggered at epoch {epoch}.")
        break

print("\n" + "=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print(f"  Best epoch:     {best_epoch}")
print(f"  Best Val AUC:   {best_metric:.4f}")
print(f"  Checkpoint:     {BEST_MODEL_PATH}")
print(f"  History:        {HISTORY_PATH}")

TRAINING — BioBERT Binary Pneumonia Classifier
  Epochs:          5
  Train batches:   210
  Primary metric:  roc_auc
  Early stopping:  patience=3

Epoch 1/5  (79s)
  Train  — Loss: 0.1805 | Acc: 89.55%
  Val    — Loss: 0.0304 | Acc: 99.50% | Recall: 0.9833 | F1: 0.9916 | AUC: 1.0000
  ✓ Best model saved — Val ROC_AUC: 1.0000

Epoch 2/5  (81s)
  Train  — Loss: 0.0314 | Acc: 99.28%
  Val    — Loss: 0.0261 | Acc: 99.50% | Recall: 0.9833 | F1: 0.9916 | AUC: 1.0000
  No improvement (1/3) — Best so far: 1.0000 (Epoch 1)

Epoch 3/5  (81s)
  Train  — Loss: 0.0223 | Acc: 99.49%
  Val    — Loss: 0.0213 | Acc: 99.50% | Recall: 0.9833 | F1: 0.9916 | AUC: 1.0000
  No improvement (2/3) — Best so far: 1.0000 (Epoch 1)


KeyboardInterrupt: 

In [9]:
# Diagnostic — find the source of perfect AUC

# 1. Check how many val texts also appear in train (should be zero)
train_texts_set = set(train_df["text_clean"])
val_texts_set   = set(val_df["text_clean"])
test_texts_set  = set(test_df["text_clean"])

print("Text overlap check:")
print(f"  Train ∩ Val:  {len(train_texts_set & val_texts_set)}")
print(f"  Train ∩ Test: {len(train_texts_set & test_texts_set)}")

# 2. Check class distribution in val more carefully
print(f"\nVal set distribution:")
print(val_df["binary_label"].value_counts())
print(f"  Pneumonia rate: {val_df['binary_label'].mean():.3f}")

# 3. Check if val is too small and too easy
# Run evaluate and look at raw probabilities
model.eval()
all_probs_val, all_labels_val = [], []
with torch.no_grad():
    for batch in val_loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["label"]
        logits         = model(input_ids, attention_mask)
        probs          = torch.softmax(logits, dim=1)[:, 1]
        all_probs_val.extend(probs.cpu().numpy())
        all_labels_val.extend(labels.numpy())

all_probs_val  = np.array(all_probs_val)
all_labels_val = np.array(all_labels_val)

print(f"\nVal probability stats:")
print(f"  Pneumonia samples    — mean prob: {all_probs_val[all_labels_val==1].mean():.4f}  "
      f"min: {all_probs_val[all_labels_val==1].min():.4f}")
print(f"  Non-pneumonia samples — mean prob: {all_probs_val[all_labels_val==0].mean():.4f}  "
      f"max: {all_probs_val[all_labels_val==0].max():.4f}")

# 4. Check if any val texts are suspiciously simple
print(f"\nSample val texts with their true label:")
for i in range(5):
    print(f"  [{all_labels_val[i]}] prob={all_probs_val[i]:.4f} | {val_df['text'].iloc[i][:70]}")

Text overlap check:
  Train ∩ Val:  0
  Train ∩ Test: 0

Val set distribution:
binary_label
0    140
1     60
Name: count, dtype: int64
  Pneumonia rate: 0.300

Val probability stats:
  Pneumonia samples    — mean prob: 0.9733  min: 0.0368
  Non-pneumonia samples — mean prob: 0.0001  max: 0.0007

Sample val texts with their true label:
  [0] prob=0.0001 | The patient presents with shortness of breath, sharp chest pain, dizzi
  [0] prob=0.0001 | The patient presents with shortness of breath, sharp chest pain, dizzi
  [0] prob=0.0001 | The patient presents with shortness of breath, sharp chest pain, dizzi
  [0] prob=0.0001 | The patient presents with shortness of breath, sharp chest pain, dizzi
  [0] prob=0.0001 | The patient presents with shortness of breath, sharp chest pain, dizzi


In [10]:
#evals


In [12]:
# Cell 7 — Load best checkpoint and full test set evaluation
from sklearn.metrics import (confusion_matrix, roc_auc_score, f1_score,
                              accuracy_score, precision_score, recall_score,
                              roc_curve)
import json

# ── Load best checkpoint ──

checkpoint = torch.load(BEST_MODEL_PATH, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)
model.eval()

print(f"Checkpoint loaded — Epoch {checkpoint['epoch']} | "
      f"Val AUC: {checkpoint['val_metrics']['roc_auc']:.4f}")

# ── Run test evaluation ──
all_preds, all_probs, all_labels = [], [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["label"]
        logits         = model(input_ids, attention_mask)
        probs          = torch.softmax(logits, dim=1)[:, 1]
        all_preds.extend(logits.argmax(dim=1).cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_probs  = np.array(all_probs)
all_labels = np.array(all_labels)

# ── Metrics ──
cm          = confusion_matrix(all_labels, all_preds)
tn, fp, fn, tp = cm.ravel()
specificity = tn / (tn + fp)

metrics = {
    "accuracy"   : round(accuracy_score(all_labels, all_preds), 4),
    "precision"  : round(precision_score(all_labels, all_preds, zero_division=0), 4),
    "recall"     : round(recall_score(all_labels, all_preds, zero_division=0), 4),
    "f1"         : round(f1_score(all_labels, all_preds, zero_division=0), 4),
    "specificity": round(specificity, 4),
    "roc_auc"    : round(roc_auc_score(all_labels, all_probs), 4),
}

print("\n" + "=" * 50)
print("TEST SET EVALUATION")
print("=" * 50)
print(f"  Samples:     {len(all_labels)} "
      f"(Pneumonia: {all_labels.sum()} | Non-pneumonia: {(all_labels==0).sum()})")
print(f"\n  Accuracy:    {metrics['accuracy']:.4f}")
print(f"  Precision:   {metrics['precision']:.4f}")
print(f"  Recall:      {metrics['recall']:.4f}   ← sensitivity")
print(f"  Specificity: {metrics['specificity']:.4f}")
print(f"  F1:          {metrics['f1']:.4f}")
print(f"  ROC-AUC:     {metrics['roc_auc']:.4f}")

print(f"\n  Confusion Matrix:")
print(f"                 Pred 0    Pred 1")
print(f"  Actual 0 (neg)   {tn:>4}      {fp:>4}   (TN / FP)")
print(f"  Actual 1 (pos)   {fn:>4}      {tp:>4}   (FN / TP)")

# ── Probability distribution ──
pneu_probs     = all_probs[all_labels == 1]
nonpneu_probs  = all_probs[all_labels == 0]

print(f"\n  Probability distribution:")
print(f"  Pneumonia     — mean: {pneu_probs.mean():.4f} | "
      f"min: {pneu_probs.min():.4f} | max: {pneu_probs.max():.4f}")
print(f"  Non-pneumonia — mean: {nonpneu_probs.mean():.4f} | "
      f"min: {nonpneu_probs.min():.4f} | max: {nonpneu_probs.max():.4f}")

# ── Uncertain predictions (0.3 < prob < 0.7) ──
uncertain_mask = (all_probs > 0.3) & (all_probs < 0.7)
print(f"\n  Uncertain predictions (0.3 < p < 0.7): {uncertain_mask.sum()}")
if uncertain_mask.sum() > 0:
    print(f"  Of those — correct: {(all_preds[uncertain_mask] == all_labels[uncertain_mask]).sum()}")

# ── Leakage confirmation on test set ──
test_in_train = set(test_df["text_clean"]) & set(train_df["text_clean"])
print(f"\n  Train ∩ Test text overlap: {len(test_in_train)}  ← confirmed no leakage")

# ── Save results ──
test_results = {
    "checkpoint_epoch" : checkpoint["epoch"],
    "test_metrics"     : metrics,
    "confusion_matrix" : cm.tolist(),
    "probability_stats": {
        "pneumonia"    : {"mean": round(float(pneu_probs.mean()), 4),
                          "min" : round(float(pneu_probs.min()),  4),
                          "max" : round(float(pneu_probs.max()),  4)},
        "non_pneumonia": {"mean": round(float(nonpneu_probs.mean()), 4),
                          "min" : round(float(nonpneu_probs.min()),  4),
                          "max" : round(float(nonpneu_probs.max()),  4)},
    },
    "uncertain_predictions": int(uncertain_mask.sum()),
    "leakage_confirmed_zero": len(test_in_train) == 0,
    "dataset_note": (
        "Near-perfect separation is attributed to the structured synthetic "
        "nature of symptom sentences generated from disease-specific binary "
        "symptom vectors, producing disease-distinctive vocabulary patterns. "
        "No leakage detected. Result reflects dataset properties, not "
        "general clinical generalisability."
    )
}

TEST_RESULTS_PATH = os.path.join(RESULTS_DIR, "test_results.json")
with open(TEST_RESULTS_PATH, "w") as f:
    json.dump(test_results, f, indent=2)

print(f"\n  Results saved to: {TEST_RESULTS_PATH}")
print("\nCell 7 complete.")

Checkpoint loaded — Epoch 1 | Val AUC: 1.0000

TEST SET EVALUATION
  Samples:     190 (Pneumonia: 58 | Non-pneumonia: 132)

  Accuracy:    0.9947
  Precision:   0.9831
  Recall:      1.0000   ← sensitivity
  Specificity: 0.9924
  F1:          0.9915
  ROC-AUC:     1.0000

  Confusion Matrix:
                 Pred 0    Pred 1
  Actual 0 (neg)    131         1   (TN / FP)
  Actual 1 (pos)      0        58   (FN / TP)

  Probability distribution:
  Pneumonia     — mean: 0.9998 | min: 0.9996 | max: 0.9998
  Non-pneumonia — mean: 0.0078 | min: 0.0001 | max: 0.9988

  Uncertain predictions (0.3 < p < 0.7): 0

  Train ∩ Test text overlap: 0  ← confirmed no leakage

  Results saved to: /content/drive/MyDrive/BioBERT_Project/biobert_pneumonia/results/test_results.json

Cell 7 complete.


In [13]:
# Cell 8 — Save tokenizer, config, experiment summary + inference function
import json, os
from transformers import BertTokenizer

# ── Save tokenizer to Drive ──
TOKENIZER_DIR = os.path.join(CHECKPOINT_DIR, "tokenizer")
tokenizer.save_pretrained(TOKENIZER_DIR)
print(f"Tokenizer saved: {TOKENIZER_DIR}")

# ── Save complete experiment summary ──
experiment_summary = {
    "experiment"        : "BioBERT Binary Pneumonia Classifier",
    "model"             : CONFIG["model_name"],
    "task"              : "Binary classification — Pneumonia (1) vs Non-pneumonia (0)",
    "dataset" : {
        "source"            : "BioBERT_Text_Dataset_Encoded.csv",
        "total_rows"        : 3766,
        "after_conflict_removal": 3739,
        "conflicts_removed" : 27,
        "pneumonia"         : 1200,
        "non_pneumonia"     : 2539,
        "imbalance_ratio"   : "2.11:1",
    },
    "splits" : {
        "train" : {"samples": 3349, "pneumonia": 1082, "non_pneumonia": 2267},
        "val"   : {"samples": 200,  "pneumonia": 60,   "non_pneumonia": 140},
        "test"  : {"samples": 190,  "pneumonia": 58,   "non_pneumonia": 132},
        "leakage": "zero — group-aware split on identical texts",
    },
    "hyperparameters" : {
        "max_length"    : CONFIG["max_length"],
        "batch_size"    : CONFIG["batch_size"],
        "learning_rate" : CONFIG["learning_rate"],
        "num_epochs"    : CONFIG["num_epochs"],
        "warmup_ratio"  : CONFIG["warmup_ratio"],
        "weight_decay"  : CONFIG["weight_decay"],
        "dropout"       : 0.3,
        "seed"          : CONFIG["seed"],
        "class_weights" : {"non_pneumonia": 0.7386, "pneumonia": 1.5476},
    },
    "training" : {
        "best_epoch"    : 1,
        "best_val_auc"  : 1.0000,
        "early_stopped" : False,
    },
    "test_metrics" : {
        "accuracy"      : 0.9947,
        "precision"     : 0.9831,
        "recall"        : 1.0000,
        "specificity"   : 0.9924,
        "f1"            : 0.9915,
        "roc_auc"       : 1.0000,
        "confusion_matrix": [[131, 1], [0, 58]],
    },
    "dataset_limitation" : (
        "Near-perfect performance is attributed to the synthetic structured "
        "nature of symptom sentences generated from disease-specific binary "
        "symptom vectors, producing highly distinctive vocabulary patterns "
        "per disease. Results reflect dataset properties and do not imply "
        "generalisation to real clinical free-text notes."
    ),
    "files" : {
        "best_checkpoint" : BEST_MODEL_PATH,
        "tokenizer"       : TOKENIZER_DIR,
        "test_results"    : os.path.join(RESULTS_DIR, "test_results.json"),
        "training_history": os.path.join(RESULTS_DIR, "training_history.json"),
    }
}

SUMMARY_PATH = os.path.join(RESULTS_DIR, "experiment_summary.json")
with open(SUMMARY_PATH, "w") as f:
    json.dump(experiment_summary, f, indent=2)
print(f"Experiment summary saved: {SUMMARY_PATH}")

# ── Verify all files exist on Drive ──
print("\nFinal file check:")
files_to_check = [
    BEST_MODEL_PATH,
    TOKENIZER_DIR,
    os.path.join(RESULTS_DIR, "test_results.json"),
    os.path.join(RESULTS_DIR, "training_history.json"),
    SUMMARY_PATH,
]
for f in files_to_check:
    exists = os.path.exists(f)
    print(f"  {'OK' if exists else 'MISSING'}   {f}")

print("\nCell 8 complete.")

Tokenizer saved: /content/drive/MyDrive/BioBERT_Project/biobert_pneumonia/checkpoints/tokenizer
Experiment summary saved: /content/drive/MyDrive/BioBERT_Project/biobert_pneumonia/results/experiment_summary.json

Final file check:
  OK   /content/drive/MyDrive/BioBERT_Project/biobert_pneumonia/checkpoints/best_model.pt
  OK   /content/drive/MyDrive/BioBERT_Project/biobert_pneumonia/checkpoints/tokenizer
  OK   /content/drive/MyDrive/BioBERT_Project/biobert_pneumonia/results/test_results.json
  OK   /content/drive/MyDrive/BioBERT_Project/biobert_pneumonia/results/training_history.json
  OK   /content/drive/MyDrive/BioBERT_Project/biobert_pneumonia/results/experiment_summary.json

Cell 8 complete.


In [15]:
# Cell 9 — Quick inference test using best checkpoint
model.eval()

test_sentences = [
    "I have fever, cough, shortness of breath, and chills.",
    "The patient presents with nausea, back pain, heartburn, and regurgitation.",
    "The is having  sharp chest pain, drug abuse, and shoulder pain.",
]

print("INFERENCE TEST — Binary Pneumonia Classifier")
print("=" * 55)

for text in test_sentences:
    encoding = tokenizer(
        text, max_length=CONFIG["max_length"],
        padding="max_length", truncation=True, return_tensors="pt"
    )
    with torch.no_grad():
        logits = model(
            encoding["input_ids"].to(device),
            encoding["attention_mask"].to(device)
        )
        probs = torch.softmax(logits, dim=1).squeeze(0).cpu().numpy()

    predicted = "Pneumonia" if probs[1] > 0.5 else "Non-pneumonia"
    print(f"\nText: {text}")
    print(f"  P(Non-pneumonia): {probs[0]:.4f}")
    print(f"  P(Pneumonia):     {probs[1]:.4f}")
    print(f"  Prediction:       {predicted}")

INFERENCE TEST — Binary Pneumonia Classifier

Text: I have fever, cough, shortness of breath, and chills.
  P(Non-pneumonia): 0.0008
  P(Pneumonia):     0.9992
  Prediction:       Pneumonia

Text: The patient presents with nausea, back pain, heartburn, and regurgitation.
  P(Non-pneumonia): 0.9998
  P(Pneumonia):     0.0002
  Prediction:       Non-pneumonia

Text: The is having  sharp chest pain, drug abuse, and shoulder pain.
  P(Non-pneumonia): 0.9995
  P(Pneumonia):     0.0005
  Prediction:       Non-pneumonia
